In [1]:
"""
CA Mental Health Services — Power BI Data Prep
Author: Kesheka Edupuganti
Source: CHHS Open Data Portal (data.chhs.ca.gov)

Run this script in the same folder as your four CHHS CSV files.
It outputs one clean CSV ready to load directly into Power BI.
"""

import pandas as pd
import numpy as np
import os

print("Loading CHHS data files...")

# ── Load raw files ────────────────────────────────────────────────────────────
sex_df  = pd.read_csv('adult_utilization_by_sex.csv')
race_df = pd.read_csv('adult_utilization_by_race.csv')
age_df  = pd.read_csv('adult_utilization_by_age.csv')
demo_df = pd.read_csv('all_demo_data_new_3.csv',
                       encoding='latin-1', engine='python', on_bad_lines='skip')

print(f"Sex  : {len(sex_df):,} rows")
print(f"Race : {len(race_df):,} rows")
print(f"Age  : {len(age_df):,} rows")
print(f"Demo : {len(demo_df):,} rows")

# ── Aggregate rows to exclude ─────────────────────────────────────────────────
EXCLUDE = ['Statewide', 'Large Counties', 'Very Large Counties (LA)',
           'Medium Counties', 'Small Counties']

def clean(df, demo_filter, label_col='Demographic Group'):
    out = df[
        (df['Population Category'] == 'Adult') &
        (df['Units'] == 'BENE_COUNT') &
        (df[label_col] == demo_filter) &
        (~df['Health Care Delivery System'].str.contains(' - ', na=False)) &
        (~df['Health Care Delivery System'].isin(EXCLUDE)) &
        (~df['Health Care Delivery System'].str.startswith('MCP-'))
    ].copy()
    out['Amount MH Service Received'] = pd.to_numeric(
        out['Amount MH Service Received'], errors='coerce'
    )
    return out.rename(columns={
        'Health Care Delivery System': 'county',
        'Fiscal Year': 'fiscal_year',
        'Amount MH Service Received': 'beneficiaries',
        'Medi-Cal Delivery System': 'delivery_system'
    })[['county', 'fiscal_year', 'delivery_system', 'beneficiaries']]


# ── 1. MAIN TABLE: county x year x delivery system ───────────────────────────
print("\nBuilding main beneficiary table...")

main = clean(sex_df, 'S0 All')
main = main.groupby(['county', 'fiscal_year', 'delivery_system'])[
    'beneficiaries'].sum().reset_index()
main = main.dropna(subset=['beneficiaries'])

# Add total across delivery systems
total = main.groupby(['county', 'fiscal_year'])[
    'beneficiaries'].sum().reset_index()
total['delivery_system'] = 'Total'
main = pd.concat([main, total], ignore_index=True)

# Year-over-year growth (Total only)
total_only = main[main['delivery_system'] == 'Total'].copy()
total_only = total_only.sort_values(['county', 'fiscal_year'])
total_only['prev_year'] = total_only.groupby('county')['beneficiaries'].shift(1)
total_only['yoy_growth_pct'] = (
    (total_only['beneficiaries'] - total_only['prev_year']) /
    total_only['prev_year'] * 100
).round(2)
total_only = total_only.drop(columns='prev_year')

# Merge growth back
main = main.merge(
    total_only[['county', 'fiscal_year', 'yoy_growth_pct']],
    on=['county', 'fiscal_year'], how='left'
)

print(f"Main table: {len(main):,} rows")


# ── 2. SEX TABLE ──────────────────────────────────────────────────────────────
print("Building sex breakdown table...")

sex_map = {'S1 Female': 'Female', 'S2 Male': 'Male'}
sex_rows = []
for code, label in sex_map.items():
    df = clean(sex_df, code)
    df = df.groupby(['county', 'fiscal_year'])['beneficiaries'].sum().reset_index()
    df['sex'] = label
    sex_rows.append(df)

sex_out = pd.concat(sex_rows, ignore_index=True).dropna(subset=['beneficiaries'])
print(f"Sex table: {len(sex_out):,} rows")


# ── 3. RACE TABLE ─────────────────────────────────────────────────────────────
print("Building race breakdown table...")

race_map = {
    'R1 Alaskan Native or American Indian': 'Native American',
    'R2 Asian or Pacific Islander':         'Asian/Pacific Islander',
    'R3 Black':                             'Black',
    'R4 Hispanic':                          'Hispanic',
    'R5 White':                             'White',
    'R6 Other':                             'Other',
    'R7 Unknown':                           'Unknown'
}
race_rows = []
for code, label in race_map.items():
    df = clean(race_df, code)
    df = df.groupby(['county', 'fiscal_year'])['beneficiaries'].sum().reset_index()
    df['race_ethnicity'] = label
    race_rows.append(df)

race_out = pd.concat(race_rows, ignore_index=True).dropna(subset=['beneficiaries'])
print(f"Race table: {len(race_out):,} rows")


# ── 4. AGE TABLE ──────────────────────────────────────────────────────────────
print("Building age breakdown table...")

age_map = {
    'A1 Adults 21-32': '21-32',
    'A2 Adults 33-44': '33-44',
    'A3 Adults 45-56': '45-56',
    'A4 Adults 57-68': '57-68',
    'A5 Adults 69+':   '69+'
}
age_rows = []
for code, label in age_map.items():
    df = clean(age_df, code)
    df = df.groupby(['county', 'fiscal_year'])['beneficiaries'].sum().reset_index()
    df['age_group'] = label
    age_rows.append(df)

age_out = pd.concat(age_rows, ignore_index=True).dropna(subset=['beneficiaries'])
print(f"Age table: {len(age_out):,} rows")


# ── 5. COUNTY SUMMARY TABLE (for KPI cards and map) ──────────────────────────
print("Building county summary table...")

summary = total_only.copy()

# Add demographic majorities per county (across all years)
# Dominant sex
sex_dominant = sex_out.groupby(['county', 'sex'])['beneficiaries'].sum().reset_index()
sex_dominant = sex_dominant.loc[
    sex_dominant.groupby('county')['beneficiaries'].idxmax()
][['county', 'sex']].rename(columns={'sex': 'dominant_sex'})

# Dominant race
race_dominant = race_out.groupby(['county', 'race_ethnicity'])['beneficiaries'].sum().reset_index()
race_dominant = race_dominant.loc[
    race_dominant.groupby('county')['beneficiaries'].idxmax()
][['county', 'race_ethnicity']].rename(columns={'race_ethnicity': 'dominant_race'})

# Dominant age
age_dominant = age_out.groupby(['county', 'age_group'])['beneficiaries'].sum().reset_index()
age_dominant = age_dominant.loc[
    age_dominant.groupby('county')['beneficiaries'].idxmax()
][['county', 'age_group']].rename(columns={'age_group': 'dominant_age_group'})

summary = summary.merge(sex_dominant,  on='county', how='left')
summary = summary.merge(race_dominant, on='county', how='left')
summary = summary.merge(age_dominant,  on='county', how='left')

print(f"Summary table: {len(summary):,} rows")


# ── Export ────────────────────────────────────────────────────────────────────
print("\nExporting CSVs...")

os.makedirs('powerbi_data', exist_ok=True)

main.to_csv('powerbi_data/01_main_beneficiaries.csv', index=False)
sex_out.to_csv('powerbi_data/02_by_sex.csv', index=False)
race_out.to_csv('powerbi_data/03_by_race.csv', index=False)
age_out.to_csv('powerbi_data/04_by_age.csv', index=False)
summary.to_csv('powerbi_data/05_county_summary.csv', index=False)

print("\nFiles saved to powerbi_data/ folder:")
for f in sorted(os.listdir('powerbi_data')):
    size = os.path.getsize(f'powerbi_data/{f}') // 1024
    print(f"  {f} ({size} KB)")

print("\nData prep complete. Load all 5 CSVs into Power BI.")
print("Tip: In Power BI use 'county' and 'fiscal_year' as your slicer fields.")

Loading CHHS data files...
Sex  : 340,839 rows
Race : 631,704 rows
Age  : 544,836 rows
Demo : 134,891 rows

Building main beneficiary table...
Main table: 708 rows
Building sex breakdown table...
Sex table: 472 rows
Building race breakdown table...
Race table: 1,627 rows
Building age breakdown table...
Age table: 1,179 rows
Building county summary table...
Summary table: 236 rows

Exporting CSVs...

Files saved to powerbi_data/ folder:
  01_main_beneficiaries.csv (23 KB)
  02_by_sex.csv (13 KB)
  03_by_race.csv (53 KB)
  04_by_age.csv (33 KB)
  05_county_summary.csv (12 KB)

Data prep complete. Load all 5 CSVs into Power BI.
Tip: In Power BI use 'county' and 'fiscal_year' as your slicer fields.
